# Chess Move Error Detection

## Problem

Each example is the first **40 plies** of a real chess game in SAN notation, where exactly
one of the first **10 plies** has been replaced by the move played at that same ply in a
*different* real game. The task is to recover the position of the substituted move, i.e. a
**10-class classification problem** over positions `1,...,10`. The moves after ply 10 are
part of the input: a corruption early in the opening leaves traces in what follows, so the
tail of the sequence carries signal about the head.

The dataset provides 400,000 training and 50,000 test examples, with three columns:
`sequence` (the 40 moves), `error_position` (the target, 1-10) and `correct_move` (the
original move). `correct_move` is never used as an input feature; it is used to reconstruct
uncorrupted games for the augmentation described below.

## Constraints

No external chess knowledge of any kind: no engines, opening books, move generators,
legality checks, or pretrained chess models. The rules of the game are treated as unknown,
and moves are handled as opaque string tokens throughout. The model must stay within
**6,000,000 trainable parameters**.

## Approach and stack

TensorFlow/Keras on Colab (GPU runtime), with experiment tracking on Weights & Biases.
The notebook runs top to bottom as a single pipeline: tokenizer → augmentation pools →
`tf.data` → model → training → evaluation.

The final model is a bidirectional GRU encoder with a **pointer head**: rather than pooling
the game into one vector and classifying it into 10 abstract classes, it keeps one hidden
state per move and scores each of the 10 candidate positions with a shared scorer. It is
trained on a mixture of real examples and examples corrupted on the fly using
**prefix-indexed candidate pools** built from the training data alone.

It reaches **0.9028** Accuracy@1 with **417,153** parameters — 7% of the allowed budget.
The full sequence of experiments, including the two that were rejected, is in the *Results*
section at the end.

## Setup

In [19]:
!pip -q install gdown wandb

In [20]:
import os
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Names this experiment: drives both the W&B run and the checkpoint filename, so
# each run keeps its own weights instead of overwriting the previous one.
RUN_NAME = "pointer-aug50"

CONFIG = {
    "seed": SEED,
    "run_name": RUN_NAME,
    "seq_len": 40,
    # model
    "embed_dim": 64,
    "gru_units": 128,
    "dense_units": 128,
    "dropout": 0.2,
    # optimisation
    "batch_size": 256,
    "epochs": 50,
    "learning_rate": 1e-3,
    "val_fraction": 0.1,
    "max_param_budget": 6_000_000,
    # augmentation - see "Prefix-indexed augmentation"
    "aug_fraction": 0.5,          # synthetic share of the augmentable block
    "aug_plies": [0, 1, 2, 3, 4, 5],   # plies 1..6, the ones with prefix coverage
    "min_pool": 8,                # minimum candidates for a group to be usable
    "word_dropout": 0.0,          # fraction of tokens replaced by [UNK]; 0.0 = off
}

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
CONFIG

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


{'seed': 42,
 'run_name': 'pointer-aug50',
 'seq_len': 40,
 'embed_dim': 64,
 'gru_units': 128,
 'dense_units': 128,
 'dropout': 0.2,
 'batch_size': 256,
 'epochs': 50,
 'learning_rate': 0.001,
 'val_fraction': 0.1,
 'max_param_budget': 6000000,
 'aug_fraction': 0.5,
 'aug_plies': [0, 1, 2, 3, 4, 5],
 'min_pool': 8,
 'word_dropout': 0.0}

## Data

In [21]:
import gdown

TRAIN_FILE_ID = "1xyggntfZ2-6BTAagxJm-tFKDXerTAs8c"
TEST_FILE_ID  = "1VozRpr3dlVpA17BL-7ALCaOXa2vCe64J"

TRAIN_PATH = "chess_error_detection_train.csv"
TEST_PATH  = "chess_error_detection_test.csv"

if not os.path.exists(TRAIN_PATH):
    gdown.download(id=TRAIN_FILE_ID, output=TRAIN_PATH, quiet=False)

if not os.path.exists(TEST_PATH):
    gdown.download(id=TEST_FILE_ID, output=TEST_PATH, quiet=False)

In [22]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)

# Assumptions the rest of the pipeline relies on: fixed 40-move sequences and
# targets in 1..10. Checked once here so that any surprise fails loudly and early.
for name, df in [("train", train_df), ("test", test_df)]:
    lengths = df["sequence"].str.split().str.len()
    assert set(df.columns) == {"sequence", "error_position", "correct_move"}
    assert (lengths == CONFIG["seq_len"]).all(), f"{name}: variable-length sequences"
    assert df["error_position"].between(1, 10).all(), f"{name}: target out of range"

display(train_df.head())

Train shape: (400000, 3)
Test shape:  (50000, 3)


,sequence,error_position,correct_move
0,e4 c6 f4 Nf6 Nc3 g6 Nf3 Bg7 d4 O-O h3 c5 d5 Nb...,2,d6
1,d4 Nf6 c4 exf4 Nc3 Bb4 Bd2 Nc6 e3 O-O Nf3 b6 d...,4,e6
2,e3 g6 d4 Bg7 c4 d6 Nf3 Nf6 Nf3 O-O O-O Nbd7 Nc...,9,Bd3
3,e4 b6 Nf3 Bb7 d4 d5 e5 e6 Bd3 Qe7 O-O fxe5 Qe1...,10,f6
4,d4 Nf6 b3 g6 Nc3 Bg7 e4 d6 f3 e5 dxe5 dxe5 Qxd...,3,c4


### Class distribution

The corrupted position is sampled uniformly over the first 10 plies, so a random
classifier scores about **10%** Accuracy@1. That is the reference to beat.

In [23]:
distribution = pd.DataFrame({
    "train": train_df["error_position"].value_counts(normalize=True).sort_index(),
    "test": test_df["error_position"].value_counts(normalize=True).sort_index(),
})

display(distribution)

,train,test
error_position,,
1,0.099727,0.09972
2,0.100695,0.10070
3,0.100165,0.10018
4,0.099255,0.09926
5,0.100720,0.10072
6,0.099965,0.09996
7,0.099625,0.09962
8,0.099903,0.09990
9,0.099748,0.09974


### One example

Printing a single game with the corrupted ply marked makes the task concrete: nothing in
the sequence looks locally impossible, which is why the problem is hard without chess
knowledge.

In [24]:
row = train_df.sample(1, random_state=SEED).iloc[0]
pos = int(row["error_position"])

for i, move in enumerate(row["sequence"].split(), start=1):
    marker = "  <-- corrupted" if i == pos else ""
    print(f"{i:2d}: {move}{marker}")

print("Correct move:", row["correct_move"])

 1: d4  <-- corrupted
 2: e5
 3: Nf3
 4: Nc6
 5: Bc4
 6: Bc5
 7: d3
 8: Nf6
 9: Nc3
10: h6
11: Be3
12: Bxe3
13: fxe3
14: Ng4
15: Qd2
16: d6
17: h3
18: Nf6
19: O-O
20: Ne7
21: Qf2
22: c6
23: Bb3
24: Ng6
25: Nh4
26: Nxh4
27: Qxh4
28: g5
29: Qg3
30: Nh5
31: Bxf7+
32: Kd7
33: Bxh5
34: Kc7
35: Bg4
36: Bxg4
37: Qxg4
38: Qc8
39: Qxc8+
40: Raxc8
Correct move: e4


## Evaluation metric

The official metric is **Accuracy@1**: a prediction counts as correct only when the
position with the highest predicted probability is exactly the corrupted one. The model
outputs a distribution over classes `0,...,9`, so `argmax` is shifted by one to match the
dataset convention `1,...,10`.

In [25]:
def accuracy_at_1(y_true, probabilities):
    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    assert probabilities.ndim == 2
    assert probabilities.shape[1] == 10
    assert len(y_true) == len(probabilities)

    y_pred = np.argmax(probabilities, axis=1) + 1

    return np.mean(y_pred == y_true)

## Train / validation split

`test_df` is left untouched for the final evaluation. The validation set is carved out of
`train_df` only, stratified on `error_position` so all 10 classes stay balanced.

In [26]:
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=CONFIG["val_fraction"],
    random_state=SEED,
    stratify=train_df["error_position"],
)

print("Train split:", train_split_df.shape)
print("Val split:  ", val_split_df.shape)
print("Test (held out):", test_df.shape)

Train split: (360000, 3)
Val split:   (40000, 3)
Test (held out): (50000, 3)


## Tokenizer

Moves are treated as opaque strings: the vocabulary is built **only from the training
split**, with no legality checks or move generators involved — just the tokens that happen
to occur in the data. `StringLookup` maps anything unseen to a single out-of-vocabulary
index, so the model still receives something for tokens it never trained on. The measured
OOV rate on validation is reported in the diagnostics section at the end.

In [27]:
def build_vocab(sequences):
    tokens = set()
    for s in sequences:
        tokens.update(s.split())
    return sorted(tokens)


vocab = build_vocab(train_split_df["sequence"])
vocab_set = set(vocab)
print("Vocabulary size (train split only):", len(vocab))

lookup = tf.keras.layers.StringLookup(vocabulary=vocab, oov_token="[UNK]")
VOCAB_SIZE = lookup.vocabulary_size()
print("StringLookup size (incl. OOV):", VOCAB_SIZE)

Vocabulary size (train split only): 3673
StringLookup size (incl. OOV): 3674


## Prefix-indexed augmentation

The baseline overfits: it reaches 0.90 training accuracy against 0.80 on validation while
using 7% of the parameter budget, so the binding constraint is generalization, not capacity.
More training data is therefore the lever — and `correct_move` makes it available. Restoring
the correct move at the corrupted ply reconstructs the *uncorrupted* game, which can then be
re-corrupted at a different position, producing a fresh example every epoch.

The question is which move to substitute in. A first attempt pooled, for each ply, all the
corrupted moves observed at that ply anywhere in the training set. Counting the distinct
moves in each pool shows why that fails:

| ply | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| distinct moves | 20 | 20 | 81 | 101 | 184 | 251 | 285 | 323 | 371 | 417 |

Twenty at plies 1 and 2 is exactly the number of moves available at the start of a game, and
the growth afterwards tracks the growing variety of reachable positions. The corruptions are
therefore **conditioned on the board position**, while a per-ply pool is the union over all
positions. Sampling from it would produce moves that are impossible in the specific game at
hand, and the model would learn to detect impossibility — an easier and different task that
does not transfer to the test set, where every corruption is a move that was really played.

Grouping by the **prefix of clean moves** instead fixes this without any notion of chess.
Two games sharing the same first `p` moves are in the same position, so any move played at
ply `p+1` by any of them is a legitimate candidate here, by construction. Candidates come
from two sources: the clean continuations of every game in the group (about 360,000 per ply)
and the corrupted moves observed there (about 36,000). The first source is what makes the
idea viable — ten times the material of the corruptions alone.

Coverage decays with depth, since prefixes fragment: by ply 10 there are 221,781 distinct
prefixes over 360,000 games and the median group offers a single candidate. The table
printed below quantifies this, and it is why `aug_plies` covers only plies 1-6. Plies 7-10
are trained on real examples only — which, as the per-position results show, still improve,
because the representations learned on the early plies transfer.

In [28]:

NUM_CAND = 10
MIN_POOL = CONFIG["min_pool"]   # minimum distinct candidates in a group, correct move included

seqs = train_split_df["sequence"].values
errs = (train_split_df["error_position"].values - 1).astype("int32")
corrects = train_split_df["correct_move"].values

clean_moves, corrupt_moves = [], []
for seq, p, correct in zip(seqs, errs, corrects):
    moves = seq.split()
    corrupt_moves.append(moves[p])   # the corrupted move, kept aside
    moves[p] = correct               # restore the clean game
    clean_moves.append(moves)


def _to_ids(list_of_moves, chunk=50_000):
    """StringLookup in blocks, to avoid materialising millions of strings at once."""
    out = []
    for i in range(0, len(list_of_moves), chunk):
        block = tf.constant(list_of_moves[i:i + chunk])
        out.append(tf.cast(lookup(block), tf.int32).numpy())
    return np.concatenate(out, axis=0)


clean_ids_np = _to_ids(clean_moves)
corrupt_ids_np = _to_ids(corrupt_moves)
N = len(clean_ids_np)
assert clean_ids_np.shape == (N, CONFIG["seq_len"])

# ---- Build one candidate pool per (ply, prefix) group --------------------
# Groups are numbered globally across plies. Candidates are stored in one flat
# array with per-group (start, length), which is what tf.data can gather from.

group_of = np.zeros((N, NUM_CAND), dtype=np.int64)
tok_blocks, start_blocks, len_blocks = [], [], []
offset = 0        # running index into the flat candidate array
group_base = 0    # running global group id

for p in range(NUM_CAND):
    if p == 0:
        inv = np.zeros(N, dtype=np.int64)   # empty prefix: every game together
        n_groups = 1
    else:
        _, inv = np.unique(clean_ids_np[:, :p], axis=0, return_inverse=True)
        inv = inv.ravel()
        n_groups = int(inv.max()) + 1

    # (group, token) pairs from both sources, deduplicated
    a = np.stack([inv, clean_ids_np[:, p].astype(np.int64)], axis=1)
    m = errs == p
    b = np.stack([inv[m], corrupt_ids_np[m].astype(np.int64)], axis=1)
    pairs = np.unique(np.concatenate([a, b], axis=0), axis=0)   # sorted by group

    counts = np.bincount(pairs[:, 0], minlength=n_groups)
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])

    tok_blocks.append(pairs[:, 1].astype(np.int32))
    start_blocks.append((starts + offset).astype(np.int32))
    len_blocks.append(counts.astype(np.int32))
    group_of[:, p] = inv + group_base

    offset += len(pairs)
    group_base += n_groups

cand_tokens_np = np.concatenate(tok_blocks)
cand_start_np = np.concatenate(start_blocks)
cand_len_np = np.concatenate(len_blocks)

cand_tokens = tf.constant(cand_tokens_np)
cand_start = tf.constant(cand_start_np)
cand_len = tf.constant(cand_len_np)

# A (game, ply) pair is usable only if its group offers enough alternatives.
pool_size = cand_len_np[group_of]              # (N, 10)
valid = pool_size >= MIN_POOL                  # (N, 10) bool

# ---- Diagnostics ---------------------------------------------------------
print(f"Clean games: {clean_ids_np.shape}")
print(f"Total candidate entries: {len(cand_tokens_np):,}\n")
print(f"{'ply':>4} {'groups':>9} {'usable games':>14} {'median pool':>12} {'max pool':>9}")
for p in range(NUM_CAND):
    ps = pool_size[:, p]
    print(f"{p+1:>4} {len(np.unique(group_of[:, p])):>9,} "
          f"{valid[:, p].mean():>13.1%} "
          f"{np.median(ps):>12.0f} {ps.max():>9}")

Clean games: (360000, 40)
Total candidate entries: 1,076,310

 ply    groups   usable games  median pool  max pool
   1         1        100.0%           20        20
   2        20        100.0%           20        20
   3       353         99.7%           51        66
   4     3,201         96.6%           46        77
   5    13,853         87.7%           31       103
   6    37,863         72.9%           18        97
   7    77,266         53.0%            8        83
   8   124,010         36.3%            4        88
   9   175,572         22.4%            2        83
  10   221,781         13.3%            1        66


## `tf.data` pipeline

Validation and test go through `make_dataset` unchanged: real examples, split on whitespace,
looked up, batched. No padding is needed since every sequence is exactly 40 moves, and
`ensure_shape` pins the static shape that `tf.strings.split` alone would leave as `(None,)`.

Training is a weighted mixture of streams. Each augmentable ply gets its own infinite stream
of on-the-fly corruptions, and the streams are weighted equally so synthetic labels stay
uniform over plies 1-6 despite their very different coverage. Real examples are then split
into an *early* block (error in plies 1-6) and a *late* block (plies 7-10), and the two
blocks are weighted 6:4 so that **every class keeps mass 1/10** whatever `aug_fraction` is.
Without that correction the model would see a skewed prior and exploit it on a uniform test
set.

`aug_fraction` therefore means: what share of the augmentable block is synthetic. At 0.5,
30% of all training examples are generated. The training set is infinite, so the epoch
length is set by hand to match the real training set size, keeping the W&B curves
comparable across runs.

In [29]:

def make_corrupt_fn(p):
    """Corrupt at a fixed ply p, sampling from that game's prefix group."""
    def fn(clean_row, g):
        start = tf.gather(cand_start, g)
        n = tf.gather(cand_len, g)

        k = tf.random.uniform([], 0, n, dtype=tf.int32)
        tok = tf.gather(cand_tokens, start + k)

        # The pool contains the correct move too; one resample makes drawing it
        # (which would produce an uncorrupted game with a false label) rare.
        k2 = tf.random.uniform([], 0, n, dtype=tf.int32)
        tok2 = tf.gather(cand_tokens, start + k2)
        tok = tf.where(tf.equal(tok, clean_row[p]), tok2, tok)

        ids = tf.tensor_scatter_nd_update(clean_row, [[p]], [tok])
        return ids, tf.constant(p, dtype=tf.int32)
    return fn


def make_ply_dataset(p):
    """Infinite stream of examples corrupted at ply p. None if coverage is zero."""
    idx = np.nonzero(valid[:, p])[0]
    if len(idx) == 0:
        return None
    ds = tf.data.Dataset.from_tensor_slices(
        (clean_ids_np[idx], group_of[idx, p].astype(np.int32))
    )
    ds = ds.shuffle(min(50_000, len(idx)), seed=SEED,
                    reshuffle_each_iteration=True).repeat()
    return ds.map(make_corrupt_fn(p), num_parallel_calls=tf.data.AUTOTUNE)


def word_dropout(ids, label):
    """Replace a fraction of tokens with [UNK] (index 0), never the corrupted one."""
    rate = CONFIG["word_dropout"]
    if rate <= 0:
        return ids, label
    drop = tf.random.uniform([CONFIG["seq_len"]]) < rate
    target = tf.one_hot(label, CONFIG["seq_len"], on_value=True,
                        off_value=False, dtype=tf.bool)
    drop = tf.logical_and(drop, tf.logical_not(target))
    return tf.where(drop, tf.zeros_like(ids), ids), label


def encode_str(seq_str, label):
    ids = tf.cast(lookup(tf.strings.split(seq_str)), tf.int32)
    return tf.ensure_shape(ids, [CONFIG["seq_len"]]), label


def make_dataset(df, batch_size=CONFIG["batch_size"], shuffle=False):
    """Unchanged from the baseline: used for val and test."""
    sequences = df["sequence"].values
    labels = (df["error_position"].values - 1).astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((sequences, labels))
    ds = ds.map(encode_str, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(10_000, seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


AUG_PLIES = CONFIG["aug_plies"]
n_early = len(AUG_PLIES)
n_late = NUM_CAND - n_early

# Split the real examples by whether their error falls in an augmentable ply.
early_mask = np.isin(errs, AUG_PLIES)


def _real_stream(mask):
    return (tf.data.Dataset
            .from_tensor_slices((seqs[mask], errs[mask]))
            .shuffle(50_000, seed=SEED, reshuffle_each_iteration=True)
            .map(encode_str, num_parallel_calls=tf.data.AUTOTUNE)
            .repeat())


def make_train_dataset(batch_size=CONFIG["batch_size"]):
    """Mix synthetic and real examples while keeping labels uniform over 1..10.

    aug_fraction now means: what share of the *augmentable block* (plies in
    AUG_PLIES) is synthetic. The late block is always real, and the two blocks
    are weighted n_early:n_late so every class keeps mass 1/10.
    """
    frac = CONFIG["aug_fraction"]
    early_w = n_early / NUM_CAND      # 0.6 with six augmentable plies
    late_w = n_late / NUM_CAND        # 0.4

    streams, weights = [], []

    if frac > 0:
        per_ply = [d for d in (make_ply_dataset(p) for p in AUG_PLIES)
                   if d is not None]
        aug_ds = tf.data.Dataset.sample_from_datasets(
            per_ply, weights=[1.0 / len(per_ply)] * len(per_ply), seed=SEED)
        streams.append(aug_ds)
        weights.append(early_w * frac)

    if frac < 1:
        streams.append(_real_stream(early_mask))
        weights.append(early_w * (1.0 - frac))

    streams.append(_real_stream(~early_mask))
    weights.append(late_w)

    ds = (streams[0] if len(streams) == 1
          else tf.data.Dataset.sample_from_datasets(streams, weights=weights,
                                                    seed=SEED))
    ds = ds.map(word_dropout, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


train_ds = make_train_dataset()
val_ds = make_dataset(val_split_df)
test_ds = make_dataset(test_df)

STEPS_PER_EPOCH = len(train_split_df) // CONFIG["batch_size"]
print("steps_per_epoch:", STEPS_PER_EPOCH)

# Labels should come out roughly uniform over 0..9.
for ids, lab in train_ds.take(1):
    print("ids:", ids.shape, ids.dtype, " labels:", lab.shape, lab.dtype)
    print("label counts in batch:", np.bincount(lab.numpy(), minlength=10))

steps_per_epoch: 1406
ids: (256, 40) <dtype: 'int32'>  labels: (256,) <dtype: 'int32'>
label counts in batch: [19 22 23 23 22 29 25 30 26 37]


## Model: BiGRU encoder with a pointer head

An embedding feeds a bidirectional GRU with `return_sequences=True`, so the encoder produces
one hidden state **per move** rather than a single pooled vector. Each state sees both
directions: the opening that precedes it and the roughly 30 moves that follow, which are the
stronger evidence — the rest of the game continues consistently with the move that was
actually played, so a substitution leaves traces downstream.

The first 10 states are kept and passed through a scorer shared across positions, producing
one scalar per candidate; the softmax is over those ten comparable scores. This differs from
a plain 10-way classifier in what it asks the network to do: "is this move plausible in its
context?", ten times with the same weights, instead of compressing the game and then
reconstructing which position was anomalous from that compressed vector. It also uses
slightly *fewer* parameters than the classifier head it replaces (417,153 against 418,314),
so the two are directly comparable.

No `mask_zero` on the embedding — index 0 is the OOV bucket, not padding, and the sequences
are fixed-length anyway.

In [30]:
def build_model(
    vocab_size,
    seq_len=CONFIG["seq_len"],
    embed_dim=CONFIG["embed_dim"],
    gru_units=CONFIG["gru_units"],
    score_units=CONFIG["dense_units"],
    dropout=CONFIG["dropout"],
    num_candidates=10,
):
    inputs = tf.keras.Input(shape=(seq_len,), dtype=tf.int32, name="move_ids")
    x = tf.keras.layers.Embedding(vocab_size, embed_dim, name="move_embedding")(inputs)

    # return_sequences=True: one hidden state per move instead of a single
    # pooled vector. The backward pass carries the 30 following moves, which is
    # the strongest evidence that a move does not fit its context.
    x = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(gru_units, return_sequences=True), name="bigru_encoder"
    )(x)

    # Drop positions 11..40: only the 10 candidates are scored.
    cand = tf.keras.layers.Cropping1D((0, seq_len - num_candidates))(x)

    # The same scorer is applied to each candidate, so the softmax is over ten
    # comparable scores rather than ten abstract classes.
    h = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(score_units, activation="relu"), name="scorer_hidden"
    )(cand)
    h = tf.keras.layers.Dropout(dropout)(h)
    scores = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(1), name="scorer_out"
    )(h)
    scores = tf.keras.layers.Flatten()(scores)
    outputs = tf.keras.layers.Softmax(name="error_position")(scores)

    return tf.keras.Model(inputs, outputs, name="chess_error_pointer")


model = build_model(VOCAB_SIZE)
model.summary()

trainable_params = int(np.sum([np.prod(v.shape) for v in model.trainable_variables]))
print(f"Trainable parameters: {trainable_params:,} / {CONFIG['max_param_budget']:,}")
assert trainable_params <= CONFIG["max_param_budget"], "Model exceeds the 6M parameter budget!"

Model: "chess_error_pointer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ move_ids (InputLayer)           │ (None, 40)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ move_embedding (Embedding)      │ (None, 40, 64)         │       235,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bigru_encoder (Bidirectional)   │ (None, 40, 256)        │       148,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cropping1d_1 (Cropping1D)       │ (None, 10, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scorer_hidden (TimeDistributed) │ (None, 10, 128)        │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scorer_out (TimeDistributed)    │ (None, 10, 1)          │           129 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ error_position (Softmax)        │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 417,153 (1.59 MB)

 Trainable params: 417,153 (1.59 MB)

 Non-trainable params: 0 (0.00 B)

Trainable parameters: 417,153 / 6,000,000


## Checkpointing

The Colab runtime is ephemeral: a timeout or a disconnection wipes everything under
`/content`, trained weights included. Checkpoints therefore go to Google Drive, keeping the
best epoch by `val_accuracy` across sessions. Set `USE_DRIVE = False` for short throwaway
runs, or if the Drive authorization flow fails — the fallback keeps the notebook runnable,
at the cost of losing the weights with the session.

In [31]:
USE_DRIVE = True

CHECKPOINT_DIR = "/content/checkpoints"

if USE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        CHECKPOINT_DIR = "/content/drive/MyDrive/chess_error_detection/checkpoints"
    except Exception as err:
        print(f"Drive mount failed ({err}) - falling back to ephemeral runtime storage.")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}.weights.h5")
print("Checkpoint path:", CHECKPOINT_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint path: /content/drive/MyDrive/chess_error_detection/checkpoints/pointer-aug50.weights.h5


## Training

Metrics are logged to Weights & Biases. `wandb.login()` asks for an API key
(https://wandb.ai/authorize) once per session.

All three callbacks monitor `val_accuracy`, not `val_loss`. The two diverge sharply here:
validation loss bottoms out around epoch 19 and then climbs as the model grows
over-confident on the examples it gets wrong, while validation accuracy keeps improving for
another twenty epochs. Since Accuracy@1 is the metric being optimised, stopping on the loss
throws away real gains — switching the monitor alone was worth +1.3 points.

In [32]:
import wandb
from wandb.integration.keras import WandbMetricsLogger

try:
    from google.colab import userdata

    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass   # falls back to the interactive prompt

wandb.login()

run = wandb.init(
    project="chess-error-detection",
    config=CONFIG,
    name=RUN_NAME,
)

In [33]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(CONFIG["learning_rate"]),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    WandbMetricsLogger(),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    # Halve the learning rate when validation accuracy stops improving.
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", mode="max",
        factor=0.5, patience=3, min_lr=1e-5, verbose=1
    ),
    # Stop once val_accuracy has plateaued, so epochs need not be tuned by hand.
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max",
        patience=6, restore_best_weights=True, verbose=1
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["epochs"],
    steps_per_epoch=STEPS_PER_EPOCH,   # required: the dataset is infinite
    callbacks=callbacks,
)

Epoch 1/50
1406/1406 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3682 - loss: 1.8008
Epoch 1: val_accuracy improved from None to 0.59940, saving model to /content/drive/MyDrive/chess_error_detection/checkpoints/pointer-aug50.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/chess_error_detection/checkpoints/pointer-aug50.weights.h5
1406/1406 ━━━━━━━━━━━━━━━━━━━━ 27s 17ms/step - accuracy: 0.4836 - loss: 1.4906 - val_accuracy: 0.5994 - val_loss: 1.1605 - learning_rate: 0.0010
Epoch 2/50
1404/1406 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6551 - loss: 1.0071
Epoch 2: val_accuracy improved from 0.59940 to 0.71272, saving model to /content/drive/MyDrive/chess_error_detection/checkpoints/pointer-aug50.weights.h5

Epoch 2: finished saving model to /content/drive/MyDrive/chess_error_detection/checkpoints/pointer-aug50.weights.h5
1406/1406 ━━━━━━━━━━━━━━━━━━━━ 23s 16ms/step - accuracy: 0.6803 - loss: 0.9295 - val_accuracy: 0.7127 - val_loss: 0.8219 - learning_rate: 

## Evaluation on the public test set

The best weights by `val_accuracy` are reloaded before evaluating, so the reported score
does not depend on whether the last epoch happened to be the best one.

In [34]:
model.load_weights(CHECKPOINT_PATH)

test_probs = model.predict(test_ds)
test_labels = test_df["error_position"].values

test_accuracy = accuracy_at_1(test_labels, test_probs)
print(f"Test Accuracy@1: {test_accuracy:.4f}")

wandb.summary["trainable_params"] = trainable_params
wandb.log({"test_accuracy_at_1": test_accuracy})
wandb.finish()

196/196 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step
Test Accuracy@1: 0.9025


epoch/accuracy,▁▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████████████
epoch/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
epoch/learning_rate,█████████████████████████████▃▃▃▃▁▁▁
epoch/loss,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▄▅▆▆▇▇▇▇▇██████████████████████████
epoch/val_loss,█▅▄▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy_at_1,▁
epoch/accuracy,0.97133
epoch/epoch,35
epoch/learning_rate,0.00025
epoch/loss,0.08824


## Diagnostics

Four checks that aggregate accuracy hides.

The **OOV rate** says whether whole-move tokenization loses information on rare moves; at
4e-05 it does not, which is why character-level tokenization was never pursued.

The **distribution of predictions** says whether the model spreads its answers across the 10
positions or collapses onto a few — a degenerate model can post a respectable accuracy while
predicting nearly the same class every time.

**Accuracy@k** separates two failure modes: localising the anomaly in the wrong region
versus localising it correctly but picking the wrong ply among near neighbours. The gap
between @1 and @3 is the headroom a better head can recover, and tracking it is what
motivated the pointer head.

**Per-position accuracy and the confusion matrix** show where the remaining errors are. They
revealed the most informative structure in this task: confusions concentrate between
positions of the **same parity** (8 with 10, 7 with 9), i.e. between moves by the same
player, and at a distance rather than between adjacent plies. The model learned the
alternating-colour structure of the game from the data alone.

In [35]:
val_tokens = [t for s in val_split_df["sequence"] for t in s.split()]
oov_rate = sum(t not in vocab_set for t in val_tokens) / len(val_tokens)
print(f"OOV rate (validation): {oov_rate:.2e}")

pred = np.argmax(test_probs, axis=1) + 1
counts = np.bincount(pred, minlength=11)[1:]
expected = len(pred) / 10

print("\nPredictions per position (expected ~{:,.0f} each):".format(expected))
for position, count in enumerate(counts, start=1):
    print(f"{position:2d}: {count:6d}  ({count / expected - 1:+.1%} vs uniform)")

OOV rate (validation): 4.31e-05

Predictions per position (expected ~5,000 each):
 1:   5082  (+1.6% vs uniform)
 2:   5151  (+3.0% vs uniform)
 3:   5097  (+1.9% vs uniform)
 4:   5104  (+2.1% vs uniform)
 5:   5223  (+4.5% vs uniform)
 6:   5181  (+3.6% vs uniform)
 7:   4816  (-3.7% vs uniform)
 8:   4787  (-4.3% vs uniform)
 9:   4816  (-3.7% vs uniform)
10:   4743  (-5.1% vs uniform)


In [36]:
y_true = np.asarray(test_labels)
y_pred = np.argmax(test_probs, axis=1) + 1

# Accuracy@k: if @3 is very high, the model localises the anomaly but misses the
# exact ply, and the fix is a head with finer local resolution.
order = np.argsort(-test_probs, axis=1) + 1
for k in (1, 2, 3):
    hit = (order[:, :k] == y_true[:, None]).any(axis=1).mean()
    print(f"Accuracy@{k}: {hit:.4f}")

# Accuracy per true position (not the distribution of predictions).
print("\nAccuracy per true position:")
for p in range(1, 11):
    m = y_true == p
    print(f"  {p:2d}: {(y_pred[m] == p).mean():.4f}  (n={m.sum()})")

# 10x10 confusion matrix, rows = true, columns = predicted.
cm = np.zeros((10, 10), dtype=int)
for t, q in zip(y_true, y_pred):
    cm[t - 1, q - 1] += 1
print("\nConfusion matrix:")
print(cm)

Accuracy@1: 0.9025
Accuracy@2: 0.9743
Accuracy@3: 0.9873

Accuracy per true position:
   1: 0.9637  (n=4986)
   2: 0.9438  (n=5035)
   3: 0.9092  (n=5009)
   4: 0.9234  (n=4963)
   5: 0.9164  (n=5036)
   6: 0.9122  (n=4998)
   7: 0.8731  (n=4981)
   8: 0.8633  (n=4995)
   9: 0.8644  (n=4987)
  10: 0.8549  (n=5010)

Confusion matrix:
[[4805    7   84   12   42    3   14    2   14    3]
 [   5 4752   19  109   16   49   10   29   13   33]
 [ 101   10 4554   18  149   12   70   10   63   22]
 [   7  135   13 4583   23   98    4   50   14   36]
 [  44   19  133   19 4615   22   88   13   58   25]
 [   7   64   20  123   24 4559   20   86   29   66]
 [  45   14  104   23  162   59 4349   49  142   34]
 [  15   52   37  100   33  180   57 4312   66  143]
 [  31   26   94   28  116   51  158   74 4311   98]
 [  22   72   39   89   43  148   46  162  106 4283]]


## Results

| | |
|---|---|
| Encoder | Embedding(64) → BiGRU(128), `return_sequences=True` |
| Head | Crop to first 10 → shared Dense(128, ReLU) → Dropout(0.2) → Dense(1) → Softmax over 10 |
| Trainable parameters | **417,153** of 6,000,000 (7%) |
| Test Accuracy@1 | **0.9028** |
| Test Accuracy@2 / @3 | 0.9730 / 0.9871 |
| Random baseline | 0.10 |

### How it got there

Each run changed one thing, so every delta is attributable.

| # | change | Test Accuracy@1 |
|---|---|---|
| 1 | BiGRU baseline, pooled vector, callbacks on `val_loss` | 0.7989 |
| 2 | callbacks monitor `val_accuracy` | 0.8120 |
| 3 | + prefix-indexed augmentation, `aug_fraction = 0.5` | 0.8578 |
| 4 | `aug_fraction = 1.0` — *rejected* | 0.8535 |
| 5 | + pointer head | **0.9028** |

Measurement noise is about ±0.4% on 50,000 test examples, so differences smaller than that
are not interpreted.

### What was rejected, and why

**Flat per-ply corruption pools.** Rejected before training, on the evidence of the distinct
move counts per ply: the pools reproduce the set of legal moves, showing corruptions are
position-conditioned, so a global pool would mostly generate moves impossible in context.

**Saturating the augmentable block (`aug_fraction = 1.0`).** Lost half a point overall, and
the per-position breakdown explains the trade-off: the augmented plies got *worse* (ply 3
dropped 5 points) while the untouched deep plies improved. Fully synthetic early examples
lose contact with the real corruption distribution exactly where opening patterns are most
specific. Half and half is the better compromise.

### Note on the parameter budget

The final model uses 7% of the 6M allowance. Growing it was considered and deliberately not
done: at 418k parameters the baseline already showed a 10-point train/validation gap, so the
binding constraint was generalization. Every point gained here came from the data pipeline
and from the structure of the output head, not from capacity.

## Next steps

- **Ensemble of 3 seeds**, averaging probabilities. The validation loss of the final model
  rises only 0.07 over the twenty epochs after its minimum, so the model stays reasonably
  calibrated and probability averaging is preferable to logit averaging. Typically worth
  one to two points at no risk.
- **Candidate-to-candidate attention** before scoring, letting each of the 10 positions see
  the other nine. This targets the parity structure still visible in the confusion matrix,
  which is the largest identifiable block of remaining errors.
- **Transformer encoder** in place of the BiGRU. With 40-token sequences the quadratic
  attention is not a concern, the timesteps parallelise (so it is faster than a deeper GRU
  at equal capacity), and every position compares directly against every other — which is
  exactly the operation this task needs.
- **Word dropout** (`word_dropout` is wired but was left at 0.0), to attack memorization of
  common opening lines specifically.
- **Extending augmentation to plies 7-10** is explicitly *not* a priority: those plies
  improved by as much as the augmented ones without receiving a single synthetic example,
  so the benefit reaches them indirectly and the sparse prefix coverage there would add
  little.